# `pg_context_expander` (PgContextExpanderModule)
- **Category**: Logic (Context Expansion)
- **Role**: RRF 상위 후보 셀이 위치한 행(Row)의 전체 열(Columns: 시계열 연도별 셀 전체)을 PostgreSQL에서 일괄 조회하여 온전한 원본 다큐먼트 문자열 리스트(`items: List[str]`)로 확장 복원합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().name == "modules" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.retrieval.context_expander import PgContextExpanderModule, PgContextExpanderInputDTO, PgContextExpanderConfigDTO

mock_pg_store = MagicMock()
mock_pg_store.fetch_rows_cells.return_value = {
    5: [
        {
            "col_index": 1,
            "column_header": ["2021"],
            "cell_value": "516339",
            "cell_coord": "B5",
            "row_header": ["영업이익"],
            "source_text": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2021 | Cell Value: 516339"
        },
        {
            "col_index": 2,
            "column_header": ["2022"],
            "cell_value": "433766",
            "cell_coord": "C5",
            "row_header": ["영업이익"],
            "source_text": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2022 | Cell Value: 433766"
        },
        {
            "col_index": 3,
            "column_header": ["2023"],
            "cell_value": "65670",
            "cell_coord": "D5",
            "row_header": ["영업이익"],
            "source_text": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: 65670"
        }
    ]
}

module = PgContextExpanderModule(pgvector_store=mock_pg_store)

sample_input = {
    "retrieval_json": {
        "query_context": {"question_id": "QUERY-001", "question_text": "2023년 삼성전자 영업이익"},
        "document_context": {"file_name": "samsung_2023.xlsx", "workbook_hash": "hash_samsung_2023", "index_id": "idx_samsung_2023"},
        "items": [
            {
                "rank": 1,
                "cell_id": "삼성전자:손익계산서:D5",
                "rrf_score": 0.0327,
                "text": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: 65670",
                "matched_subquery": "Company: 삼성전자 | Sheet: 손익계산서 | Row Header: 영업이익 | Column Header: 2023 | Cell Value: ?"
            }
        ]
    }
}
input_dto = PgContextExpanderInputDTO(**sample_input)
output = module.run(input_dto, config=PgContextExpanderConfigDTO(top_k=5, max_blocks=100))
print_io("pg_context_expander (PgContextExpanderModule)", sample_input, output)
